In [1]:
import kagglehub

path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

print("Dataset path:", path)

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset path: /kaggle/input/skin-cancer9-classesisic


In [3]:
DATASET_PATH = "/kaggle/input/skin-cancer9-classesisic"

print(DATASET_PATH)

/kaggle/input/skin-cancer9-classesisic


In [4]:
import os

print("Dataset contents:\n")

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    if level >= 2:
        continue

    for file in files[:10]:
        print(f"{indent}  {file}")

Dataset contents:

skin-cancer9-classesisic/
  Skin cancer ISIC The International Skin Imaging Collaboration/
    Test/
      pigmented benign keratosis/
      melanoma/
      vascular lesion/
      actinic keratosis/
      squamous cell carcinoma/
      basal cell carcinoma/
      seborrheic keratosis/
      dermatofibroma/
      nevus/
    Train/
      pigmented benign keratosis/
      melanoma/
      vascular lesion/
      actinic keratosis/
      squamous cell carcinoma/
      basal cell carcinoma/
      seborrheic keratosis/
      dermatofibroma/
      nevus/


In [5]:
import os
import numpy as np
import pandas as pd

DATASET_PATH = "/kaggle/input/skin-cancer9-classesisic"

print("Dataset path:", DATASET_PATH)
print("Exists:", os.path.exists(DATASET_PATH))

Dataset path: /kaggle/input/skin-cancer9-classesisic
Exists: True


In [6]:
for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, "").count(os.sep)

    if level <= 2:
        print("\n" + "  " * level + os.path.basename(root) + "/")

        for file in files[:10]:
            print("  " * (level + 1) + file)


skin-cancer9-classesisic/

  Skin cancer ISIC The International Skin Imaging Collaboration/

    Test/

    Train/


In [7]:
import os

DATASET_ROOT = "/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration"

TRAIN_DIR = os.path.join(DATASET_ROOT, "Train")
TEST_DIR = os.path.join(DATASET_ROOT, "Test")

print("TRAIN CLASSES:")
print(os.listdir(TRAIN_DIR))

print("\nTEST CLASSES:")
print(os.listdir(TEST_DIR))

TRAIN CLASSES:
['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']

TEST CLASSES:
['pigmented benign keratosis', 'melanoma', 'vascular lesion', 'actinic keratosis', 'squamous cell carcinoma', 'basal cell carcinoma', 'seborrheic keratosis', 'dermatofibroma', 'nevus']


In [8]:
from collections import Counter

def count_images(folder):
    result = {}

    for class_name in sorted(os.listdir(folder)):
        class_path = os.path.join(folder, class_name)

        if os.path.isdir(class_path):
            images = [
                f for f in os.listdir(class_path)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
            result[class_name] = len(images)

    return result


train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("TRAIN IMAGE COUNTS:")
for cls, count in train_counts.items():
    print(f"{cls}: {count}")

print("\nTEST IMAGE COUNTS:")
for cls, count in test_counts.items():
    print(f"{cls}: {count}")

TRAIN IMAGE COUNTS:
actinic keratosis: 114
basal cell carcinoma: 376
dermatofibroma: 95
melanoma: 438
nevus: 357
pigmented benign keratosis: 462
seborrheic keratosis: 77
squamous cell carcinoma: 181
vascular lesion: 139

TEST IMAGE COUNTS:
actinic keratosis: 16
basal cell carcinoma: 16
dermatofibroma: 16
melanoma: 16
nevus: 16
pigmented benign keratosis: 16
seborrheic keratosis: 3
squamous cell carcinoma: 16
vascular lesion: 3


In [9]:
SELECTED_CLASSES = [
    "pigmented benign keratosis",
    "melanoma",
    "nevus",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

print("Selected classes:")
for i, cls in enumerate(SELECTED_CLASSES, 1):
    print(f"{i}. {cls}")

Selected classes:
1. pigmented benign keratosis
2. melanoma
3. nevus
4. basal cell carcinoma
5. squamous cell carcinoma


In [10]:
import os
import numpy as np
from PIL import Image

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

def get_image_paths_and_labels(folder, selected_classes):
    image_paths = []
    labels = []

    for label, class_name in enumerate(selected_classes):
        class_folder = os.path.join(folder, class_name)

        for file_name in os.listdir(class_folder):
            if file_name.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append(os.path.join(class_folder, file_name))
                labels.append(label)

    return np.array(image_paths), np.array(labels)


train_paths, train_labels = get_image_paths_and_labels(
    TRAIN_DIR,
    SELECTED_CLASSES
)

test_paths, test_labels = get_image_paths_and_labels(
    TEST_DIR,
    SELECTED_CLASSES
)

print("Training images:", len(train_paths))
print("Testing images:", len(test_paths))
print("Number of classes:", len(SELECTED_CLASSES))

Training images: 1814
Testing images: 80
Number of classes: 5


In [11]:
from sklearn.model_selection import train_test_split

# 80% training, 20% validation
train_paths_split, val_paths, train_labels_split, val_labels = train_test_split(
    train_paths,
    train_labels,
    test_size=0.20,
    random_state=42,
    stratify=train_labels
)

print("Training images:", len(train_paths_split))
print("Validation images:", len(val_paths))
print("Testing images:", len(test_paths))

Training images: 1451
Validation images: 363
Testing images: 80


In [12]:
from collections import Counter

print("Training distribution:")
for label, count in sorted(Counter(train_labels_split).items()):
    print(f"{SELECTED_CLASSES[label]}: {count}")

print("\nValidation distribution:")
for label, count in sorted(Counter(val_labels).items()):
    print(f"{SELECTED_CLASSES[label]}: {count}")

print("\nTest distribution:")
for label, count in sorted(Counter(test_labels).items()):
    print(f"{SELECTED_CLASSES[label]}: {count}")

Training distribution:
pigmented benign keratosis: 369
melanoma: 350
nevus: 286
basal cell carcinoma: 301
squamous cell carcinoma: 145

Validation distribution:
pigmented benign keratosis: 93
melanoma: 88
nevus: 71
basal cell carcinoma: 75
squamous cell carcinoma: 36

Test distribution:
pigmented benign keratosis: 16
melanoma: 16
nevus: 16
basal cell carcinoma: 16
squamous cell carcinoma: 16


In [13]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU is available")
else:
    print("⚠️ GPU is NOT available")

TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
✅ GPU is available


In [14]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = len(SELECTED_CLASSES)

# Training augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Validation/Test: only normalization
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0
)

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)

Image size: (224, 224)
Batch size: 32
Number of classes: 5


In [24]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# EfficientNet-B0 ke liye
train_datagen_eff = ImageDataGenerator(
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    fill_mode="nearest"
)

# Validation/Test mein augmentation nahi
val_test_datagen_eff = ImageDataGenerator()


train_generator_eff = train_datagen_eff.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_eff = val_test_datagen_eff.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_eff = val_test_datagen_eff.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

print("EfficientNet generators ready.")

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.
EfficientNet generators ready.


In [25]:
from tensorflow.keras.applications import EfficientNetB0

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

efficientnet_base.trainable = False

efficientnet_model = keras.Sequential([
    efficientnet_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

efficientnet_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,384,936 (16.73 MB)

 Trainable params: 332,293 (1.27 MB)

 Non-trainable params: 4,052,643 (15.46 MB)

In [15]:
def create_dataframe(image_paths, labels):
    return pd.DataFrame({
        "filename": image_paths,
        "class": [SELECTED_CLASSES[label] for label in labels]
    })


train_df = create_dataframe(train_paths_split, train_labels_split)
val_df = create_dataframe(val_paths, val_labels)
test_df = create_dataframe(test_paths, test_labels)

train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [16]:
print("Class indices:")
print(train_generator.class_indices)

images, labels = next(train_generator)

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Class indices:
{'pigmented benign keratosis': 0, 'melanoma': 1, 'nevus': 2, 'basal cell carcinoma': 3, 'squamous cell carcinoma': 4}

Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 5)


In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import time
import numpy as np
import pandas as pd

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Training settings
EPOCHS = 15

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

print("Training configuration ready.")
print("Epochs:", EPOCHS)
print("Classes:", NUM_CLASSES)

Training configuration ready.
Epochs: 15
Classes: 5


In [19]:
def evaluate_model(model, test_generator, model_name):

    # Reset generator
    test_generator.reset()

    # Prediction start time
    start_time = time.time()

    y_prob = model.predict(
        test_generator,
        verbose=1
    )

    inference_time = time.time() - start_time

    # Predicted labels
    y_pred = np.argmax(y_prob, axis=1)

    # True labels
    y_true = test_generator.classes

    # Metrics
    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # One-hot labels for multiclass AUC
    y_true_onehot = tf.keras.utils.to_categorical(
        y_true,
        num_classes=NUM_CLASSES
    )

    auc = roc_auc_score(
        y_true_onehot,
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

    results = {
        "Model": model_name,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100,
        "AUC (%)": auc * 100,
        "Inference Time (s)": inference_time
    }

    return results

In [20]:
def build_transfer_model(base_model, model_name):

    # Freeze pretrained layers initially
    base_model.trainable = False

    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(
            256,
            activation="relu"
        ),
        layers.Dropout(0.3),
        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ], name=model_name)

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=0.0001
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [21]:
from tensorflow.keras.applications import EfficientNetB0

efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

efficientnet_model = build_transfer_model(
    efficientnet_base,
    "EfficientNet-B0"
)

efficientnet_model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "EfficientNet-B0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,378,792 (16.70 MB)

 Trainable params: 329,221 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [26]:
history_eff = efficientnet_model.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 83s 1s/step - accuracy: 0.4259 - loss: 1.7716 - val_accuracy: 0.3747 - val_loss: 1.4164 - learning_rate: 0.0010
Epoch 2/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 792ms/step - accuracy: 0.5396 - loss: 1.2944 - val_accuracy: 0.4821 - val_loss: 1.2543 - learning_rate: 0.0010
Epoch 3/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 757ms/step - accuracy: 0.5858 - loss: 1.1059 - val_accuracy: 0.5234 - val_loss: 1.1374 - learning_rate: 0.0010
Epoch 4/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 773ms/step - accuracy: 0.6010 - loss: 1.0801 - val_accuracy: 0.5730 - val_loss: 1.0904 - learning_rate: 0.0010
Epoch 5/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 765ms/step - accuracy: 0.6423 - loss: 0.9790 - val_accuracy: 0.5647 - val_loss: 1.0853 - learning_rate: 0.0010
Epoch 6/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 34s 739ms/step - accuracy: 0.6368 - loss: 0.9328 - val_accuracy: 0.5730 - val_loss: 1.0887 - learning_rate: 0.0010
Epoch 7/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 762ms/step - accuracy: 0.6423 - loss: 0.940

In [27]:
efficientnet_base.trainable = True

freeze_until = int(len(efficientnet_base.layers) * 0.70)

for layer in efficientnet_base.layers[:freeze_until]:
    layer.trainable = False

for layer in efficientnet_base.layers[freeze_until:]:
    layer.trainable = True

efficientnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning configured.")

Fine-tuning configured.


In [28]:
history_eff_ft = efficientnet_model.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 109s 2s/step - accuracy: 0.5327 - loss: 1.3077 - val_accuracy: 0.6006 - val_loss: 1.0947 - learning_rate: 1.0000e-05
Epoch 2/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 667ms/step - accuracy: 0.5227 - loss: 1.2788
Epoch 2: ReduceLROnPlateau reducing learning rate to 2.9999999242136253e-06.
46/46 ━━━━━━━━━━━━━━━━━━━━ 34s 750ms/step - accuracy: 0.5176 - loss: 1.3125 - val_accuracy: 0.5565 - val_loss: 1.1610 - learning_rate: 1.0000e-05
Epoch 3/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 832ms/step - accuracy: 0.5314 - loss: 1.3007 - val_accuracy: 0.5289 - val_loss: 1.2080 - learning_rate: 3.0000e-06
Epoch 4/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 704ms/step - accuracy: 0.5419 - loss: 1.3029
Epoch 4: ReduceLROnPlateau reducing learning rate to 8.999999636216671e-07.
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 787ms/step - accuracy: 0.5555 - loss: 1.2501 - val_accuracy: 0.5152 - val_loss: 1.2490 - learning_rate: 3.0000e-06
Epoch 4: early stopping
Restoring model weights from the end of the bes

In [30]:
efficientnet_result = evaluate_model(
    efficientnet_model,
    test_generator_eff,
    "EfficientNet-B0"
)

print("\nFinal EfficientNet-B0 Results:")

for key, value in efficientnet_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step

Final EfficientNet-B0 Results:
Model: EfficientNet-B0
Accuracy (%): 45.00
Precision (%): 50.35
Recall (%): 45.00
F1-Score (%): 41.11
AUC (%): 83.18
Inference Time (s): 3.02


In [31]:
# Validation evaluation
val_generator_eff.reset()

val_prob = efficientnet_model.predict(
    val_generator_eff,
    verbose=1
)

val_pred = np.argmax(val_prob, axis=1)
val_true = val_generator_eff.classes

val_accuracy = accuracy_score(val_true, val_pred)

print("Final model Validation Accuracy:",
      round(val_accuracy * 100, 2), "%")

12/12 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step
Final model Validation Accuracy: 60.06 %


In [32]:
from sklearn.metrics import confusion_matrix

print("VALIDATION CONFUSION MATRIX")
print(confusion_matrix(val_true, val_pred))

print("\nTEST CONFUSION MATRIX")

test_generator_eff.reset()

test_prob = efficientnet_model.predict(
    test_generator_eff,
    verbose=1
)

test_pred = np.argmax(test_prob, axis=1)
test_true = test_generator_eff.classes

print(confusion_matrix(test_true, test_pred))

VALIDATION CONFUSION MATRIX
[[81  2  5  5  0]
 [23 47 15  2  1]
 [23  5 36  7  0]
 [23  3  1 48  0]
 [19  3  2  6  6]]

TEST CONFUSION MATRIX
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step
[[10  0  2  3  1]
 [ 2  4 10  0  0]
 [ 3  0 13  0  0]
 [ 6  0  1  8  1]
 [11  1  0  3  1]]


In [33]:
from sklearn.metrics import classification_report

print(classification_report(
    test_true,
    test_pred,
    target_names=SELECTED_CLASSES,
    digits=4,
    zero_division=0
))

                            precision    recall  f1-score   support

pigmented benign keratosis     0.3125    0.6250    0.4167        16
                  melanoma     0.8000    0.2500    0.3810        16
                     nevus     0.5000    0.8125    0.6190        16
      basal cell carcinoma     0.5714    0.5000    0.5333        16
   squamous cell carcinoma     0.3333    0.0625    0.1053        16

                  accuracy                         0.4500        80
                 macro avg     0.5035    0.4500    0.4111        80
              weighted avg     0.5035    0.4500    0.4111        80



In [37]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_split),
    y=train_labels_split
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

print("Class weights:")
for i, weight in class_weights.items():
    print(f"{SELECTED_CLASSES[i]}: {weight:.3f}")

Class weights:
pigmented benign keratosis: 0.786
melanoma: 0.829
nevus: 1.015
basal cell carcinoma: 0.964
squamous cell carcinoma: 2.001


In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_split),
    y=train_labels_split
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

print("Class weights:")
for i, weight in class_weights.items():
    print(f"{SELECTED_CLASSES[i]}: {weight:.3f}")

Class weights:
pigmented benign keratosis: 0.786
melanoma: 0.829
nevus: 1.015
basal cell carcinoma: 0.964
squamous cell carcinoma: 2.001


In [39]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0

# Create a fresh EfficientNet-B0
efficientnet_base = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained backbone
efficientnet_base.trainable = False

# Build complete model
efficientnet_model_v2 = keras.Sequential([
    efficientnet_base,
    layers.GlobalAveragePooling2D(),

    layers.BatchNormalization(),
    layers.Dropout(0.4),

    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(5, activation="softmax")
], name="EfficientNetB0_Improved")

# Compile
efficientnet_model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("EfficientNet-B0 model created successfully!")
print("Model name:", efficientnet_model_v2.name)

EfficientNet-B0 model created successfully!
Model name: EfficientNetB0_Improved


In [40]:
callbacks_head = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history_head = efficientnet_model_v2.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=7,
    class_weight=class_weights,
    callbacks=callbacks_head,
    verbose=1
)

Epoch 1/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.3873 - loss: 1.8045 - val_accuracy: 0.3884 - val_loss: 1.4000 - learning_rate: 0.0010
Epoch 2/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 789ms/step - accuracy: 0.5052 - loss: 1.3915 - val_accuracy: 0.5399 - val_loss: 1.1865 - learning_rate: 0.0010
Epoch 3/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 43s 944ms/step - accuracy: 0.5844 - loss: 1.1472 - val_accuracy: 0.5647 - val_loss: 1.1120 - learning_rate: 0.0010
Epoch 4/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 781ms/step - accuracy: 0.5906 - loss: 1.0970 - val_accuracy: 0.5455 - val_loss: 1.1303 - learning_rate: 0.0010
Epoch 5/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 686ms/step - accuracy: 0.6143 - loss: 1.0047
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 781ms/step - accuracy: 0.6023 - loss: 1.0462 - val_accuracy: 0.5372 - val_loss: 1.1563 - learning_rate: 0.0010
Epoch 6/7
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 788ms/step - accuracy: 0.6258 - loss: 0.9988 - val_

In [41]:
# Unfreeze the backbone
efficientnet_base.trainable = True

# Freeze most layers, fine-tune only the last ~30%
fine_tune_from = int(len(efficientnet_base.layers) * 0.70)

for layer in efficientnet_base.layers[:fine_tune_from]:
    layer.trainable = False

for layer in efficientnet_base.layers[fine_tune_from:]:
    # Keep BatchNorm layers frozen for stability
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

print("Total layers:", len(efficientnet_base.layers))
print("Fine-tuning from layer:", fine_tune_from)

trainable_count = sum(
    1 for layer in efficientnet_base.layers
    if layer.trainable
)

print("Trainable backbone layers:", trainable_count)

Total layers: 238
Fine-tuning from layer: 166
Trainable backbone layers: 57


In [42]:
efficientnet_model_v2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_finetune = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

history_finetune = efficientnet_model_v2.fit(
    train_generator_eff,
    validation_data=val_generator_eff,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_finetune,
    verbose=1
)

Epoch 1/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 98s 1s/step - accuracy: 0.6478 - loss: 0.9033 - val_accuracy: 0.6116 - val_loss: 1.0159 - learning_rate: 1.0000e-05
Epoch 2/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 804ms/step - accuracy: 0.6520 - loss: 0.8891 - val_accuracy: 0.6061 - val_loss: 1.0058 - learning_rate: 1.0000e-05
Epoch 3/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 800ms/step - accuracy: 0.6823 - loss: 0.8628 - val_accuracy: 0.6171 - val_loss: 0.9985 - learning_rate: 1.0000e-05
Epoch 4/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 797ms/step - accuracy: 0.6857 - loss: 0.8204 - val_accuracy: 0.6336 - val_loss: 0.9942 - learning_rate: 1.0000e-05
Epoch 5/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 771ms/step - accuracy: 0.6726 - loss: 0.8354 - val_accuracy: 0.6364 - val_loss: 0.9885 - learning_rate: 1.0000e-05
Epoch 6/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 801ms/step - accuracy: 0.6768 - loss: 0.8452 - val_accuracy: 0.6391 - val_loss: 0.9771 - learning_rate: 1.0000e-05
Epoch 7/15
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 791ms/step - accura

In [43]:
efficientnet_result_v2 = evaluate_model(
    efficientnet_model_v2,
    test_generator_eff,
    "EfficientNet-B0"
)

print("\nImproved EfficientNet-B0 Results:")

for key, value in efficientnet_result_v2.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 16s 5s/step

Improved EfficientNet-B0 Results:
Model: EfficientNet-B0
Accuracy (%): 62.50
Precision (%): 71.62
Recall (%): 62.50
F1-Score (%): 60.19
AUC (%): 89.53
Inference Time (s): 16.72


VGG16 model

In [46]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers
from tensorflow import keras

vgg16_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

vgg16_base.trainable = False

vgg16_model = keras.Sequential([
    vgg16_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
], name="VGG16")

vgg16_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("VGG16 model created successfully!")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
VGG16 model created successfully!


In [47]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen_vgg = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.20,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen_vgg = ImageDataGenerator(
    rescale=1.0/255.0
)

train_generator_vgg = train_datagen_vgg.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_vgg = val_test_datagen_vgg.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_vgg = val_test_datagen_vgg.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [48]:
callbacks_vgg = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history_vgg16 = vgg16_model.fit(
    train_generator_vgg,
    validation_data=val_generator_vgg,
    epochs=5,
    class_weight=class_weights,
    callbacks=callbacks_vgg,
    verbose=1
)

Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.3177 - loss: 1.9383 - val_accuracy: 0.2424 - val_loss: 1.6806 - learning_rate: 0.0010
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 815ms/step - accuracy: 0.3970 - loss: 1.6292 - val_accuracy: 0.2452 - val_loss: 1.6159 - learning_rate: 0.0010
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 817ms/step - accuracy: 0.4335 - loss: 1.5446 - val_accuracy: 0.2755 - val_loss: 1.5436 - learning_rate: 0.0010
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 821ms/step - accuracy: 0.4514 - loss: 1.4367 - val_accuracy: 0.3113 - val_loss: 1.5196 - learning_rate: 0.0010
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 812ms/step - accuracy: 0.4859 - loss: 1.3555 - val_accuracy: 0.3416 - val_loss: 1.4937 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 5.


In [49]:
# Unfreeze VGG16 backbone
vgg16_base.trainable = True

# Freeze first ~70% layers
fine_tune_from = int(len(vgg16_base.layers) * 0.70)

for layer in vgg16_base.layers[:fine_tune_from]:
    layer.trainable = False

for layer in vgg16_base.layers[fine_tune_from:]:
    # Keep BatchNorm frozen if present
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

print("Total VGG16 layers:", len(vgg16_base.layers))
print("Fine-tuning from layer:", fine_tune_from)

trainable_layers = sum(
    1 for layer in vgg16_base.layers
    if layer.trainable
)

print("Trainable backbone layers:", trainable_layers)

Total VGG16 layers: 19
Fine-tuning from layer: 13
Trainable backbone layers: 6


In [50]:
vgg16_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_vgg_ft = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [51]:
history_vgg16_ft = vgg16_model.fit(
    train_generator_vgg,
    validation_data=val_generator_vgg,
    epochs=5,
    class_weight=class_weights,
    callbacks=callbacks_vgg_ft,
    verbose=1
)

Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 58s 991ms/step - accuracy: 0.4976 - loss: 1.3405 - val_accuracy: 0.4160 - val_loss: 1.3511 - learning_rate: 1.0000e-05
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 840ms/step - accuracy: 0.5362 - loss: 1.2193 - val_accuracy: 0.4931 - val_loss: 1.2522 - learning_rate: 1.0000e-05
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 830ms/step - accuracy: 0.5314 - loss: 1.1761 - val_accuracy: 0.5262 - val_loss: 1.1778 - learning_rate: 1.0000e-05
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 839ms/step - accuracy: 0.5713 - loss: 1.1230 - val_accuracy: 0.5399 - val_loss: 1.1306 - learning_rate: 1.0000e-05
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 817ms/step - accuracy: 0.5975 - loss: 1.0827 - val_accuracy: 0.5317 - val_loss: 1.1083 - learning_rate: 1.0000e-05
Restoring model weights from the end of the best epoch: 5.


In [52]:
vgg16_result = evaluate_model(
    vgg16_model,
    test_generator_vgg,
    "VGG16"
)

print("\nFinal VGG16 Results:")

for key, value in vgg16_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 12s 5s/step

Final VGG16 Results:
Model: VGG16
Accuracy (%): 51.25
Precision (%): 63.07
Recall (%): 51.25
F1-Score (%): 48.04
AUC (%): 81.80
Inference Time (s): 13.03


In [53]:
from sklearn.metrics import confusion_matrix, classification_report

test_generator_vgg.reset()

y_prob_vgg = vgg16_model.predict(
    test_generator_vgg,
    verbose=1
)

y_pred_vgg = np.argmax(y_prob_vgg, axis=1)
y_true_vgg = test_generator_vgg.classes

print("\nClassification Report:")
print(classification_report(
    y_true_vgg,
    y_pred_vgg,
    target_names=SELECTED_CLASSES,
    digits=4,
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true_vgg, y_pred_vgg))

3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step

Classification Report:
                            precision    recall  f1-score   support

pigmented benign keratosis     0.4545    0.6250    0.5263        16
                  melanoma     1.0000    0.1250    0.2222        16
                     nevus     0.5000    0.8750    0.6364        16
      basal cell carcinoma     0.7778    0.4375    0.5600        16
   squamous cell carcinoma     0.4211    0.5000    0.4571        16

                  accuracy                         0.5125        80
                 macro avg     0.6307    0.5125    0.4804        80
              weighted avg     0.6307    0.5125    0.4804        80


Confusion Matrix:
[[10  0  0  1  5]
 [ 1  2 13  0  0]
 [ 2  0 14  0  0]
 [ 3  0  0  7  6]
 [ 6  0  1  1  8]]


VGG19 model

In [55]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# VGG19 backbone
vgg19_base = VGG19(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

vgg19_base.trainable = False

# VGG19 model
vgg19_model = keras.Sequential([
    vgg19_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
], name="VGG19")

vgg19_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Data generators
train_datagen_vgg19 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True
)

val_test_datagen_vgg19 = ImageDataGenerator(
    rescale=1./255
)

train_generator_vgg19 = train_datagen_vgg19.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_vgg19 = val_test_datagen_vgg19.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_vgg19 = val_test_datagen_vgg19.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

print("VGG19 model and generators created successfully!")

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.
VGG19 model and generators created successfully!


In [56]:
history_vgg19 = vgg19_model.fit(
    train_generator_vgg19,
    validation_data=val_generator_vgg19,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 49s 922ms/step - accuracy: 0.2936 - loss: 2.0026 - val_accuracy: 0.2645 - val_loss: 1.5903 - learning_rate: 0.0010
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 832ms/step - accuracy: 0.3928 - loss: 1.6588 - val_accuracy: 0.2452 - val_loss: 1.5437 - learning_rate: 0.0010
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 840ms/step - accuracy: 0.4259 - loss: 1.5707 - val_accuracy: 0.2507 - val_loss: 1.5058 - learning_rate: 0.0010
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 828ms/step - accuracy: 0.4369 - loss: 1.4483 - val_accuracy: 0.3609 - val_loss: 1.4726 - learning_rate: 0.0010
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 821ms/step - accuracy: 0.4742 - loss: 1.3548 - val_accuracy: 0.3774 - val_loss: 1.4479 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 5.


In [57]:
# Unfreeze VGG19
vgg19_base.trainable = True

# Fine-tune only last 30% layers
fine_tune_from = int(len(vgg19_base.layers) * 0.70)

for layer in vgg19_base.layers[:fine_tune_from]:
    layer.trainable = False

for layer in vgg19_base.layers[fine_tune_from:]:
    layer.trainable = True

# Recompile with small learning rate
vgg19_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_vgg19_ft = vgg19_model.fit(
    train_generator_vgg19,
    validation_data=val_generator_vgg19,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 55s 991ms/step - accuracy: 0.4914 - loss: 1.3073 - val_accuracy: 0.5289 - val_loss: 1.3419
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 860ms/step - accuracy: 0.5445 - loss: 1.1884 - val_accuracy: 0.4711 - val_loss: 1.3019
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 43s 942ms/step - accuracy: 0.5651 - loss: 1.1253 - val_accuracy: 0.5317 - val_loss: 1.1924
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 872ms/step - accuracy: 0.6037 - loss: 1.0091 - val_accuracy: 0.5620 - val_loss: 1.1315
Epoch 5/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 42s 917ms/step - accuracy: 0.5982 - loss: 1.0789 - val_accuracy: 0.5372 - val_loss: 1.1243
Restoring model weights from the end of the best epoch: 5.


In [58]:
vgg19_result = evaluate_model(
    vgg19_model,
    test_generator_vgg19,
    "VGG19"
)

print("\nFinal VGG19 Results:")

for key, value in vgg19_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step

Final VGG19 Results:
Model: VGG19
Accuracy (%): 48.75
Precision (%): 49.90
Recall (%): 48.75
F1-Score (%): 48.34
AUC (%): 83.96
Inference Time (s): 5.60


ResNet18 model

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def residual_block(x, filters, stride=1):
    shortcut = x

    x = layers.Conv2D(
        filters, 3, strides=stride, padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters, 3, strides=1, padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(
            filters, 1, strides=stride,
            padding="same", use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


def build_resnet18(input_shape=(224, 224, 3), num_classes=5):

    inputs = keras.Input(shape=input_shape)

    x = layers.Conv2D(
        64, 7, strides=2, padding="same",
        use_bias=False
    )(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(
        pool_size=3, strides=2, padding="same"
    )(x)

    # ResNet18: 2, 2, 2, 2 blocks
    x = residual_block(x, 64, 1)
    x = residual_block(x, 64, 1)

    x = residual_block(x, 128, 2)
    x = residual_block(x, 128, 1)

    x = residual_block(x, 256, 2)
    x = residual_block(x, 256, 1)

    x = residual_block(x, 512, 2)
    x = residual_block(x, 512, 1)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="ResNet18")


resnet18_model = build_resnet18()

resnet18_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("ResNet18 created successfully!")

ResNet18 created successfully!


In [7]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Dataset paths
DATASET_ROOT = "/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration"
TRAIN_DIR = os.path.join(DATASET_ROOT, "Train")
TEST_DIR = os.path.join(DATASET_ROOT, "Test")

# Same 5 classes used for ALL models
SELECTED_CLASSES = [
    "pigmented benign keratosis",
    "melanoma",
    "nevus",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

NUM_CLASSES = 5
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# Get image paths and labels
def get_image_paths_and_labels(folder, selected_classes):
    image_paths = []
    labels = []

    for label, class_name in enumerate(selected_classes):
        class_folder = os.path.join(folder, class_name)

        for file_name in os.listdir(class_folder):
            if file_name.lower().endswith(IMAGE_EXTENSIONS):
                image_paths.append(
                    os.path.join(class_folder, file_name)
                )
                labels.append(label)

    return np.array(image_paths), np.array(labels)

train_paths, train_labels = get_image_paths_and_labels(
    TRAIN_DIR, SELECTED_CLASSES
)

test_paths, test_labels = get_image_paths_and_labels(
    TEST_DIR, SELECTED_CLASSES
)

# Same 80/20 stratified split
train_paths_split, val_paths, train_labels_split, val_labels = train_test_split(
    train_paths,
    train_labels,
    test_size=0.20,
    random_state=42,
    stratify=train_labels
)

# Create DataFrames
def create_dataframe(image_paths, labels):
    return pd.DataFrame({
        "filename": image_paths,
        "class": [
            SELECTED_CLASSES[label]
            for label in labels
        ]
    })

train_df = create_dataframe(
    train_paths_split, train_labels_split
)

val_df = create_dataframe(
    val_paths, val_labels
)

test_df = create_dataframe(
    test_paths, test_labels
)

# Class weights
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_split),
    y=train_labels_split
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

print("Dataset recovered successfully!")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Classes:", SELECTED_CLASSES)
print("Class weights:", class_weights)

Dataset recovered successfully!
Training: 1451
Validation: 363
Test: 80
Classes: ['pigmented benign keratosis', 'melanoma', 'nevus', 'basal cell carcinoma', 'squamous cell carcinoma']
Class weights: {0: 0.786449864498645, 1: 0.8291428571428572, 2: 1.0146853146853148, 3: 0.9641196013289036, 4: 2.0013793103448276}


In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen_r18 = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen_r18 = ImageDataGenerator(
    rescale=1./255
)

train_generator_r18 = train_datagen_r18.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_r18 = val_test_datagen_r18.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_r18 = val_test_datagen_r18.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

print("ResNet18 generators ready!")

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.
ResNet18 generators ready!


In [10]:
# Make sure ResNet18 exists
if "resnet18_model" not in globals():
    print("Creating ResNet18 model...")

    resnet18_model = build_resnet18(
        input_shape=(224, 224, 3),
        num_classes=5
    )

    resnet18_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

# Fast training
history_r18 = resnet18_model.fit(
    train_generator_r18,
    validation_data=val_generator_r18,
    epochs=5,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.3618 - loss: 1.8656 - val_accuracy: 0.1846 - val_loss: 2.5248 - learning_rate: 0.0010
Epoch 2/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 770ms/step - accuracy: 0.4466 - loss: 1.4811 - val_accuracy: 0.0992 - val_loss: 2.1209 - learning_rate: 0.0010
Epoch 3/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 671ms/step - accuracy: 0.4479 - loss: 1.3009
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 35s 754ms/step - accuracy: 0.4425 - loss: 1.3250 - val_accuracy: 0.1873 - val_loss: 2.5850 - learning_rate: 0.0010
Epoch 4/5
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 696ms/step - accuracy: 0.4578 - loss: 1.2307
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 779ms/step - accuracy: 0.4762 - loss: 1.2338 - val_accuracy: 0.0992 - val_loss: 2.4091 - learning_rate: 5.0000e-04
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 2.


In [12]:
import time
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, test_generator, model_name):
    test_generator.reset()

    start_time = time.time()
    y_prob = model.predict(test_generator, verbose=1)
    inference_time = time.time() - start_time

    y_pred = np.argmax(y_prob, axis=1)
    y_true = test_generator.classes

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    y_true_onehot = tf.keras.utils.to_categorical(
        y_true, num_classes=5
    )

    auc = roc_auc_score(
        y_true_onehot,
        y_prob,
        multi_class="ovr",
        average="weighted"
    )

    results = {
        "Model": model_name,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100,
        "AUC (%)": auc * 100,
        "Inference Time (s)": inference_time
    }

    return results

In [13]:
resnet18_result = evaluate_model(
    resnet18_model,
    test_generator_r18,
    "ResNet18"
)

print("\nFinal ResNet18 Results:")

for key, value in resnet18_result.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step

Final ResNet18 Results:
Model: ResNet18
Accuracy (%): 20.00
Precision (%): 4.00
Recall (%): 20.00
F1-Score (%): 6.67
AUC (%): 44.53
Inference Time (s): 7.80


In [14]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import time
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ImageNet pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT

resnet18_transfer = models.resnet18(weights=weights)

# Replace final layer for our 5 classes
resnet18_transfer.fc = nn.Linear(
    resnet18_transfer.fc.in_features,
    5
)

resnet18_transfer = resnet18_transfer.to(device)

print("Pretrained ResNet18 loaded successfully!")

Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 228MB/s]


Pretrained ResNet18 loaded successfully!


In [15]:
class SkinDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = self.df.iloc[idx]["filename"]
        class_name = self.df.iloc[idx]["class"]

        image = Image.open(image_path).convert("RGB")
        label = SELECTED_CLASSES.index(class_name)

        if self.transform:
            image = self.transform(image)

        return image, label


train_transform_r18 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform_r18 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset_r18 = SkinDataset(train_df, train_transform_r18)
val_dataset_r18 = SkinDataset(val_df, test_transform_r18)
test_dataset_r18 = SkinDataset(test_df, test_transform_r18)

train_loader_r18 = DataLoader(
    train_dataset_r18,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_r18 = DataLoader(
    val_dataset_r18,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_r18 = DataLoader(
    test_dataset_r18,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset_r18))
print("Validation:", len(val_dataset_r18))
print("Test:", len(test_dataset_r18))

Train: 1451
Validation: 363
Test: 80


In [16]:
# Freeze all pretrained layers
for param in resnet18_transfer.parameters():
    param.requires_grad = False

# Train only final FC layer
for param in resnet18_transfer.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    resnet18_transfer.fc.parameters(),
    lr=1e-3
)

print("Training ResNet18 classifier head...")

Training ResNet18 classifier head...


In [17]:
num_epochs = 5

best_val_acc = 0.0
best_state = None

for epoch in range(num_epochs):

    # ---------------- TRAIN ----------------
    resnet18_transfer.train()

    train_correct = 0
    train_total = 0

    for images, labels in train_loader_r18:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = resnet18_transfer(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = train_correct / train_total

    # ---------------- VALIDATION ----------------
    resnet18_transfer.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader_r18:

            images = images.to(device)
            labels = labels.to(device)

            outputs = resnet18_transfer(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {
            k: v.cpu().clone()
            for k, v in resnet18_transfer.state_dict().items()
        }

# Restore best model
resnet18_transfer.load_state_dict(best_state)
resnet18_transfer = resnet18_transfer.to(device)

print("\nBest Validation Accuracy:", best_val_acc * 100)

Epoch 1/5 | Train Acc: 0.4211 | Val Acc: 0.5510
Epoch 2/5 | Train Acc: 0.5693 | Val Acc: 0.5978
Epoch 3/5 | Train Acc: 0.5934 | Val Acc: 0.5950
Epoch 4/5 | Train Acc: 0.6237 | Val Acc: 0.6171
Epoch 5/5 | Train Acc: 0.6402 | Val Acc: 0.5950

Best Validation Accuracy: 61.70798898071625


In [18]:
resnet18_transfer.eval()

y_true = []
y_pred = []
y_prob = []

start_time = time.time()

with torch.no_grad():

    for images, labels in test_loader_r18:

        images = images.to(device)

        outputs = resnet18_transfer(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())
        y_prob.extend(probabilities.cpu().numpy())

inference_time = time.time() - start_time

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    y_true, y_pred,
    average="weighted",
    zero_division=0
)

y_true_onehot = np.eye(5)[y_true]

auc = roc_auc_score(
    y_true_onehot,
    y_prob,
    multi_class="ovr",
    average="weighted"
)

resnet18_result = {
    "Model": "ResNet18",
    "Accuracy (%)": accuracy * 100,
    "Precision (%)": precision * 100,
    "Recall (%)": recall * 100,
    "F1-Score (%)": f1 * 100,
    "AUC (%)": auc * 100,
    "Inference Time (s)": inference_time
}

print("\n========== RESNET18 FINAL RESULTS ==========")

for key, value in resnet18_result.items():

    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")


========== RESNET18 FINAL RESULTS ==========
Model: ResNet18
Accuracy (%): 60.00
Precision (%): 61.79
Recall (%): 60.00
F1-Score (%): 55.51
AUC (%): 86.23
Inference Time (s): 3.66


ResNet50 model

In [23]:
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen_r50 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen_r50 = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator_r50 = train_datagen_r50.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_r50 = val_test_datagen_r50.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_r50 = val_test_datagen_r50.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224,224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [24]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50

resnet50_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

resnet50_base.trainable = False

resnet50_model = keras.Sequential([
    resnet50_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
])

resnet50_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [26]:
history_r50 = resnet50_model.fit(
    train_generator_r50,
    validation_data=val_generator_r50,
    epochs=8,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step - accuracy: 0.4928 - loss: 1.6469 - val_accuracy: 0.5592 - val_loss: 1.2863 - learning_rate: 0.0010
Epoch 2/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 798ms/step - accuracy: 0.6327 - loss: 1.0557 - val_accuracy: 0.6061 - val_loss: 1.0549 - learning_rate: 0.0010
Epoch 3/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 794ms/step - accuracy: 0.6492 - loss: 0.9538 - val_accuracy: 0.6309 - val_loss: 1.0014 - learning_rate: 0.0010
Epoch 4/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 782ms/step - accuracy: 0.6823 - loss: 0.8670 - val_accuracy: 0.6749 - val_loss: 0.8808 - learning_rate: 0.0010
Epoch 5/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 699ms/step - accuracy: 0.6972 - loss: 0.8118
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 790ms/step - accuracy: 0.7002 - loss: 0.8186 - val_accuracy: 0.6694 - val_loss: 0.8856 - learning_rate: 0.0010
Epoch 6/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 36s 781ms/step - accuracy: 0.7312 - loss: 0.7278 - val_

In [27]:
resnet50_result = evaluate_model(
    resnet50_model,
    test_generator_r50,
    "ResNet50"
)

print("\n========== RESNET50 FINAL RESULTS ==========")

for key, value in resnet50_result.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

2/3 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step

3/3 ━━━━━━━━━━━━━━━━━━━━ 10s 3s/step

========== RESNET50 FINAL RESULTS ==========
Model: ResNet50
Accuracy (%): 57.50
Precision (%): 61.14
Recall (%): 57.50
F1-Score (%): 55.72
AUC (%): 89.06
Inference Time (s): 10.62


ResNet101 model

In [28]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet101

resnet101_base = ResNet101(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

resnet101_base.trainable = False

resnet101_model = keras.Sequential([
    resnet101_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
], name="ResNet101")

resnet101_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("ResNet101 pretrained model loaded successfully!")

171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step
ResNet101 pretrained model loaded successfully!


In [29]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet import preprocess_input

train_datagen_r101 = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen_r101 = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator_r101 = train_datagen_r101.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_r101 = val_test_datagen_r101.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_r101 = val_test_datagen_r101.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.


In [30]:
history_r101 = resnet101_model.fit(
    train_generator_r101,
    validation_data=val_generator_r101,
    epochs=8,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - accuracy: 0.4769 - loss: 1.6052 - val_accuracy: 0.4986 - val_loss: 1.5702 - learning_rate: 0.0010
Epoch 2/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 835ms/step - accuracy: 0.6154 - loss: 1.0837 - val_accuracy: 0.5675 - val_loss: 1.2072 - learning_rate: 0.0010
Epoch 3/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 837ms/step - accuracy: 0.6437 - loss: 1.0091 - val_accuracy: 0.5730 - val_loss: 1.0917 - learning_rate: 0.0010
Epoch 4/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 39s 844ms/step - accuracy: 0.6465 - loss: 0.9345 - val_accuracy: 0.6226 - val_loss: 0.9769 - learning_rate: 0.0010
Epoch 5/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 758ms/step - accuracy: 0.7118 - loss: 0.8292
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 40s 868ms/step - accuracy: 0.6961 - loss: 0.8331 - val_accuracy: 0.6281 - val_loss: 0.9964 - learning_rate: 0.0010
Epoch 6/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 832ms/step - accuracy: 0.7167 - loss: 0.7757 - val_

In [31]:
resnet101_result = evaluate_model(
    resnet101_model,
    test_generator_r101,
    "ResNet101"
)

print("\n========== RESNET101 FINAL RESULTS ==========")

for key, value in resnet101_result.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 17s 5s/step

========== RESNET101 FINAL RESULTS ==========
Model: ResNet101
Accuracy (%): 52.50
Precision (%): 58.62
Recall (%): 52.50
F1-Score (%): 50.11
AUC (%): 85.10
Inference Time (s): 17.09


DenseNet121 model

In [32]:
# ============================================
# DENSENET121 - IMAGENET PRETRAINED
# ============================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121

densenet_base = DenseNet121(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained backbone
densenet_base.trainable = False

densenet_model = keras.Sequential([
    densenet_base,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation="softmax")
], name="DenseNet121")

densenet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("DenseNet121 loaded successfully!")

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
DenseNet121 loaded successfully!


In [33]:
# ============================================
# DENSENET121 GENERATORS
# ============================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.densenet import preprocess_input

train_datagen_dn = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_test_datagen_dn = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator_dn = train_datagen_dn.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=True,
    seed=42
)

val_generator_dn = val_test_datagen_dn.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_generator_dn = val_test_datagen_dn.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

print("DenseNet121 generators ready!")

Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.
DenseNet121 generators ready!


In [34]:
# ============================================
# TRAIN DENSENET121
# ============================================

history_dn = densenet_model.fit(
    train_generator_dn,
    validation_data=val_generator_dn,
    epochs=8,
    class_weight=class_weights,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True,
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=1,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

Epoch 1/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.4466 - loss: 1.6206 - val_accuracy: 0.5014 - val_loss: 1.1716 - learning_rate: 0.0010
Epoch 2/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 806ms/step - accuracy: 0.5658 - loss: 1.1992 - val_accuracy: 0.5785 - val_loss: 1.0743 - learning_rate: 0.0010
Epoch 3/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 701ms/step - accuracy: 0.6533 - loss: 1.0287
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 815ms/step - accuracy: 0.6313 - loss: 1.0530 - val_accuracy: 0.5840 - val_loss: 1.0772 - learning_rate: 0.0010
Epoch 4/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 38s 803ms/step - accuracy: 0.6299 - loss: 0.9865 - val_accuracy: 0.5978 - val_loss: 1.0288 - learning_rate: 5.0000e-04
Epoch 5/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 789ms/step - accuracy: 0.6795 - loss: 0.9059 - val_accuracy: 0.6033 - val_loss: 1.0091 - learning_rate: 5.0000e-04
Epoch 6/8
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 809ms/step - accuracy: 0.6816 - loss: 0.88

In [35]:
# ============================================
# FINAL DENSENET121 RESULTS
# ============================================

densenet_result = evaluate_model(
    densenet_model,
    test_generator_dn,
    "DenseNet121"
)

print("\n========== DENSENET121 FINAL RESULTS ==========")

for key, value in densenet_result.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

3/3 ━━━━━━━━━━━━━━━━━━━━ 36s 9s/step

========== DENSENET121 FINAL RESULTS ==========
Model: DenseNet121
Accuracy (%): 55.00
Precision (%): 58.21
Recall (%): 55.00
F1-Score (%): 51.20
AUC (%): 86.43
Inference Time (s): 36.13


AlexNet

In [36]:
# ============================================
# ALEXNET - IMAGENET PRETRAINED
# ============================================

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Load ImageNet pretrained AlexNet
alexnet = models.alexnet(
    weights=models.AlexNet_Weights.DEFAULT
)

# Replace final classifier for 5 classes
alexnet.classifier[6] = nn.Linear(
    alexnet.classifier[6].in_features,
    5
)

alexnet = alexnet.to(device)

print("Pretrained AlexNet loaded successfully!")

Device: cuda
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:06<00:00, 38.4MB/s]


Pretrained AlexNet loaded successfully!


In [37]:
# ============================================
# ALEXNET DATASET
# ============================================

class SkinDatasetAlexNet(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image_path = self.df.iloc[idx]["filename"]
        class_name = self.df.iloc[idx]["class"]

        image = Image.open(image_path).convert("RGB")

        label = SELECTED_CLASSES.index(class_name)

        if self.transform:
            image = self.transform(image)

        return image, label


# ImageNet normalization
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform_alex = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])

test_transform_alex = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=imagenet_mean,
        std=imagenet_std
    )
])


train_dataset_alex = SkinDatasetAlexNet(
    train_df,
    transform=train_transform_alex
)

val_dataset_alex = SkinDatasetAlexNet(
    val_df,
    transform=test_transform_alex
)

test_dataset_alex = SkinDatasetAlexNet(
    test_df,
    transform=test_transform_alex
)


train_loader_alex = DataLoader(
    train_dataset_alex,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_alex = DataLoader(
    val_dataset_alex,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_alex = DataLoader(
    test_dataset_alex,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Train:", len(train_dataset_alex))
print("Validation:", len(val_dataset_alex))
print("Test:", len(test_dataset_alex))

Train: 1451
Validation: 363
Test: 80


In [38]:
# ============================================
# CLASS WEIGHTS
# ============================================

class_counts = np.bincount(
    train_labels_split,
    minlength=5
)

total_samples = len(train_labels_split)

weights = total_samples / (
    5 * class_counts
)

class_weights_alex = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

print("Class counts:", class_counts)
print("Class weights:", class_weights_alex)

Class counts: [369 350 286 301 145]
Class weights: tensor([0.7864, 0.8291, 1.0147, 0.9641, 2.0014], device='cuda:0')


In [39]:
# ============================================
# FREEZE PRETRAINED FEATURES
# ============================================

for param in alexnet.parameters():
    param.requires_grad = False

# Train only classifier
for param in alexnet.classifier.parameters():
    param.requires_grad = True

criterion_alex = nn.CrossEntropyLoss(
    weight=class_weights_alex
)

optimizer_alex = torch.optim.Adam(
    alexnet.classifier.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("AlexNet ready for transfer learning!")

AlexNet ready for transfer learning!


In [40]:
# ============================================
# TRAIN ALEXNET
# ============================================

num_epochs = 8

best_val_acc = 0.0
best_state = None

for epoch in range(num_epochs):

    # ---------------- TRAIN ----------------
    alexnet.train()

    train_correct = 0
    train_total = 0
    train_loss_total = 0.0

    for images, labels in train_loader_alex:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_alex.zero_grad()

        outputs = alexnet(images)

        loss = criterion_alex(outputs, labels)

        loss.backward()
        optimizer_alex.step()

        train_loss_total += loss.item() * labels.size(0)

        predictions = torch.argmax(outputs, dim=1)

        train_total += labels.size(0)
        train_correct += (
            predictions == labels
        ).sum().item()

    train_loss = train_loss_total / train_total
    train_acc = train_correct / train_total

    # ---------------- VALIDATION ----------------
    alexnet.eval()

    val_correct = 0
    val_total = 0
    val_loss_total = 0.0

    with torch.no_grad():

        for images, labels in val_loader_alex:

            images = images.to(device)
            labels = labels.to(device)

            outputs = alexnet(images)

            loss = criterion_alex(outputs, labels)

            val_loss_total += loss.item() * labels.size(0)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            val_total += labels.size(0)

            val_correct += (
                predictions == labels
            ).sum().item()

    val_loss = val_loss_total / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}%"
    )

    # Save best validation model
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        best_state = {
            k: v.cpu().clone()
            for k, v in alexnet.state_dict().items()
        }

# Restore best model
alexnet.load_state_dict(best_state)
alexnet = alexnet.to(device)

print(
    f"\nBest Validation Accuracy: "
    f"{best_val_acc*100:.2f}%"
)

Epoch 1/8 | Train Loss: 1.2914 | Train Acc: 48.31% | Val Loss: 1.2401 | Val Acc: 42.98%
Epoch 2/8 | Train Loss: 1.0253 | Train Acc: 61.54% | Val Loss: 0.9769 | Val Acc: 61.98%
Epoch 3/8 | Train Loss: 0.8895 | Train Acc: 65.89% | Val Loss: 0.9656 | Val Acc: 63.91%
Epoch 4/8 | Train Loss: 0.8468 | Train Acc: 69.12% | Val Loss: 0.9081 | Val Acc: 66.67%
Epoch 5/8 | Train Loss: 0.7932 | Train Acc: 70.92% | Val Loss: 0.9406 | Val Acc: 60.06%
Epoch 6/8 | Train Loss: 0.8042 | Train Acc: 69.33% | Val Loss: 0.9781 | Val Acc: 60.33%
Epoch 7/8 | Train Loss: 0.7626 | Train Acc: 72.02% | Val Loss: 0.9982 | Val Acc: 53.99%
Epoch 8/8 | Train Loss: 0.7134 | Train Acc: 72.98% | Val Loss: 0.8503 | Val Acc: 65.56%

Best Validation Accuracy: 66.67%


In [41]:
# ============================================
# FINAL ALEXNET RESULTS
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

alexnet.eval()

y_true_alex = []
y_pred_alex = []
y_prob_alex = []

start_time = time.time()

with torch.no_grad():

    for images, labels in test_loader_alex:

        images = images.to(device)

        outputs = alexnet(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        y_true_alex.extend(
            labels.numpy()
        )

        y_pred_alex.extend(
            predictions.cpu().numpy()
        )

        y_prob_alex.extend(
            probabilities.cpu().numpy()
        )

inference_time_alex = time.time() - start_time

y_true_alex = np.array(y_true_alex)
y_pred_alex = np.array(y_pred_alex)
y_prob_alex = np.array(y_prob_alex)

accuracy_alex = accuracy_score(
    y_true_alex,
    y_pred_alex
)

precision_alex = precision_score(
    y_true_alex,
    y_pred_alex,
    average="weighted",
    zero_division=0
)

recall_alex = recall_score(
    y_true_alex,
    y_pred_alex,
    average="weighted",
    zero_division=0
)

f1_alex = f1_score(
    y_true_alex,
    y_pred_alex,
    average="weighted",
    zero_division=0
)

y_true_onehot_alex = np.eye(5)[y_true_alex]

auc_alex = roc_auc_score(
    y_true_onehot_alex,
    y_prob_alex,
    multi_class="ovr",
    average="weighted"
)

alexnet_result = {
    "Model": "AlexNet",
    "Accuracy (%)": accuracy_alex * 100,
    "Precision (%)": precision_alex * 100,
    "Recall (%)": recall_alex * 100,
    "F1-Score (%)": f1_alex * 100,
    "AUC (%)": auc_alex * 100,
    "Inference Time (s)": inference_time_alex
}

print("\n========== ALEXNET FINAL RESULTS ==========")

for key, value in alexnet_result.items():

    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")


========== ALEXNET FINAL RESULTS ==========
Model: AlexNet
Accuracy (%): 56.25
Precision (%): 57.64
Recall (%): 56.25
F1-Score (%): 54.94
AUC (%): 83.48
Inference Time (s): 4.55


Table 1. Comparison of Transfer Learning Models

In [1]:
# ============================================================
# TABLE 1: COMPARISON OF TRANSFER LEARNING MODELS
# ============================================================

import pandas as pd
from IPython.display import display, HTML

# ------------------------------------------------------------
# FINAL RESULTS OF ALL 8 MODELS
# ------------------------------------------------------------

table1_results = {

    "AlexNet": {
        "Accuracy (%)": 56.25,
        "Precision (%)": 57.64,
        "Recall (%)": 56.25,
        "F1-Score (%)": 54.94,
        "AUC (%)": 83.48
    },

    "VGG16": {
        "Accuracy (%)": 51.25,
        "Precision (%)": 63.07,
        "Recall (%)": 51.25,
        "F1-Score (%)": 48.04,
        "AUC (%)": 81.80
    },

    "VGG19": {
        "Accuracy (%)": 48.75,
        "Precision (%)": 49.90,
        "Recall (%)": 48.75,
        "F1-Score (%)": 48.34,
        "AUC (%)": 83.96
    },

    "ResNet18": {
        "Accuracy (%)": 60.00,
        "Precision (%)": 61.79,
        "Recall (%)": 60.00,
        "F1-Score (%)": 55.51,
        "AUC (%)": 86.23
    },

    "ResNet50": {
        "Accuracy (%)": 57.50,
        "Precision (%)": 61.14,
        "Recall (%)": 57.50,
        "F1-Score (%)": 55.72,
        "AUC (%)": 89.06
    },

    "ResNet101": {
        "Accuracy (%)": 52.50,
        "Precision (%)": 58.62,
        "Recall (%)": 52.50,
        "F1-Score (%)": 50.11,
        "AUC (%)": 85.10
    },

    "DenseNet121": {
        "Accuracy (%)": 55.00,
        "Precision (%)": 58.21,
        "Recall (%)": 55.00,
        "F1-Score (%)": 51.20,
        "AUC (%)": 86.43
    },

    "EfficientNet-B0": {
        "Accuracy (%)": 62.50,
        "Precision (%)": 71.62,
        "Recall (%)": 62.50,
        "F1-Score (%)": 60.19,
        "AUC (%)": 89.53
    }
}

# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

table1_df = pd.DataFrame(table1_results).T

# Keep exact column order from assignment
table1_df = table1_df[
    [
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1-Score (%)",
        "AUC (%)"
    ]
]

# ------------------------------------------------------------
# ROUND VALUES TO 2 DECIMAL PLACES
# ------------------------------------------------------------

table1_df = table1_df.round(2)

# ------------------------------------------------------------
# DISPLAY PROFESSIONAL TABLE
# ------------------------------------------------------------

print("=" * 95)
print("TABLE 1. COMPARISON OF TRANSFER LEARNING MODELS")
print("=" * 95)

display(
    table1_df.style
    .format("{:.2f}")
    .set_caption("Table 1. Comparison of Transfer Learning Models")
    .set_properties(**{
        "text-align": "center",
        "font-size": "12pt"
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("font-size", "16pt"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "10px")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "8px")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("padding", "8px")
            ]
        }
    ])
)

# ------------------------------------------------------------
# SAVE AS CSV
# ------------------------------------------------------------

table1_df.to_csv(
    "Table_1_Comparison_of_Transfer_Learning_Models.csv"
)

print("\nTable 1 saved successfully!")
print("File: Table_1_Comparison_of_Transfer_Learning_Models.csv")

# ------------------------------------------------------------
# SHOW BEST MODEL
# ------------------------------------------------------------

best_accuracy_model = table1_df["Accuracy (%)"].idxmax()
best_auc_model = table1_df["AUC (%)"].idxmax()
best_f1_model = table1_df["F1-Score (%)"].idxmax()

print("\n========== BEST RESULTS ==========")
print(
    f"Best Accuracy : {best_accuracy_model} "
    f"({table1_df.loc[best_accuracy_model, 'Accuracy (%)']:.2f}%)"
)

print(
    f"Best AUC      : {best_auc_model} "
    f"({table1_df.loc[best_auc_model, 'AUC (%)']:.2f}%)"
)

print(
    f"Best F1-Score : {best_f1_model} "
    f"({table1_df.loc[best_f1_model, 'F1-Score (%)']:.2f}%)"
)

TABLE 1. COMPARISON OF TRANSFER LEARNING MODELS


,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
AlexNet,56.25,57.64,56.25,54.94,83.48
VGG16,51.25,63.07,51.25,48.04,81.80
VGG19,48.75,49.90,48.75,48.34,83.96
ResNet18,60.00,61.79,60.00,55.51,86.23
ResNet50,57.50,61.14,57.50,55.72,89.06
ResNet101,52.50,58.62,52.50,50.11,85.10
DenseNet121,55.00,58.21,55.00,51.20,86.43
EfficientNet-B0,62.50,71.62,62.50,60.19,89.53



Table 1 saved successfully!
File: Table_1_Comparison_of_Transfer_Learning_Models.csv

========== BEST RESULTS ==========
Best Accuracy : EfficientNet-B0 (62.50%)
Best AUC      : EfficientNet-B0 (89.53%)
Best F1-Score : EfficientNet-B0 (60.19%)


EfficientNet-B0 se Deep Features Extract

In [5]:
import os

print("=== /kaggle/input ===")

if os.path.exists("/kaggle/input"):
    print(os.listdir("/kaggle/input"))
else:
    print("/kaggle/input DOES NOT EXIST")

print("\n=== /kaggle ===")
if os.path.exists("/kaggle"):
    print(os.listdir("/kaggle"))

print("\n=== CURRENT DIRECTORY ===")
print(os.getcwd())

=== /kaggle/input ===
[]

=== /kaggle ===
['input']

=== CURRENT DIRECTORY ===
/content


In [6]:
import kagglehub
import os

# Download / locate dataset
path = kagglehub.dataset_download(
    "nodoubttome/skin-cancer9-classesisic"
)

print("Dataset path:")
print(path)

print("\nFiles/Folders:")
print(os.listdir(path))

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset path:
/kaggle/input/skin-cancer9-classesisic

Files/Folders:
['Skin cancer ISIC The International Skin Imaging Collaboration']


In [7]:
# ============================================================
# TABLE 2 - STEP 1
# PREPARE DATASET FOR DEEP FEATURE EXTRACTION
# ============================================================

import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# EXACT DATASET PATH
# ------------------------------------------------------------

DATASET_ROOT = "/kaggle/input/skin-cancer9-classesisic/Skin cancer ISIC The International Skin Imaging Collaboration"

TRAIN_DIR = os.path.join(DATASET_ROOT, "Train")
TEST_DIR = os.path.join(DATASET_ROOT, "Test")

print("Train folder exists:", os.path.exists(TRAIN_DIR))
print("Test folder exists :", os.path.exists(TEST_DIR))


# ------------------------------------------------------------
# ONLY 5 CLASSES
# ------------------------------------------------------------

SELECTED_CLASSES = [
    "pigmented benign keratosis",
    "melanoma",
    "nevus",
    "basal cell carcinoma",
    "squamous cell carcinoma"
]

NUM_CLASSES = 5
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


# ------------------------------------------------------------
# CHECK FOLDERS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CHECKING SELECTED CLASSES")
print("=" * 60)

for class_name in SELECTED_CLASSES:

    train_class_path = os.path.join(TRAIN_DIR, class_name)
    test_class_path = os.path.join(TEST_DIR, class_name)

    print("\n", class_name)
    print("  Train:", os.path.exists(train_class_path))
    print("  Test :", os.path.exists(test_class_path))


# ------------------------------------------------------------
# GET IMAGE PATHS AND LABELS
# ------------------------------------------------------------

def get_image_paths_and_labels(folder, selected_classes):

    image_paths = []
    labels = []

    for label, class_name in enumerate(selected_classes):

        class_folder = os.path.join(folder, class_name)

        for file_name in os.listdir(class_folder):

            if file_name.lower().endswith(IMAGE_EXTENSIONS):

                image_paths.append(
                    os.path.join(class_folder, file_name)
                )

                labels.append(label)

    return np.array(image_paths), np.array(labels)


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

train_paths, train_labels = get_image_paths_and_labels(
    TRAIN_DIR,
    SELECTED_CLASSES
)

test_paths, test_labels = get_image_paths_and_labels(
    TEST_DIR,
    SELECTED_CLASSES
)


print("\nTotal original training images:", len(train_paths))
print("Total test images:", len(test_paths))


# ------------------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ------------------------------------------------------------

train_paths_split, val_paths, train_labels_split, val_labels = train_test_split(
    train_paths,
    train_labels,
    test_size=0.20,
    random_state=42,
    stratify=train_labels
)


# ------------------------------------------------------------
# CREATE DATAFRAMES
# ------------------------------------------------------------

def create_dataframe(image_paths, labels):

    return pd.DataFrame({
        "filename": image_paths,
        "class": [
            SELECTED_CLASSES[int(label)]
            for label in labels
        ]
    })


train_df = create_dataframe(
    train_paths_split,
    train_labels_split
)

val_df = create_dataframe(
    val_paths,
    val_labels
)

test_df = create_dataframe(
    test_paths,
    test_labels
)


# ------------------------------------------------------------
# FINAL CHECK
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET READY")
print("=" * 60)

print("Training   :", len(train_df))
print("Validation :", len(val_df))
print("Testing    :", len(test_df))

print("\nDataFrame columns:")
print(train_df.columns.tolist())

print("\nAll 5 classes:")
for i, cls in enumerate(SELECTED_CLASSES):
    print(f"{i}: {cls}")

print("\n" + "=" * 60)
print("READY FOR EFFICIENTNET-B0 FEATURES")
print("=" * 60)

Train folder exists: True
Test folder exists : True

CHECKING SELECTED CLASSES

 pigmented benign keratosis
  Train: True
  Test : True

 melanoma
  Train: True
  Test : True

 nevus
  Train: True
  Test : True

 basal cell carcinoma
  Train: True
  Test : True

 squamous cell carcinoma
  Train: True
  Test : True

Total original training images: 1814
Total test images: 80

DATASET READY
Training   : 1451
Validation : 363
Testing    : 80

DataFrame columns:
['filename', 'class']

All 5 classes:
0: pigmented benign keratosis
1: melanoma
2: nevus
3: basal cell carcinoma
4: squamous cell carcinoma

READY FOR EFFICIENTNET-B0 FEATURES


In [9]:
# ============================================================
# TABLE 2 - EFFICIENTNET-B0 DEEP FEATURE EXTRACTION
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("=" * 60)
print("EFFICIENTNET-B0 DEEP FEATURE EXTRACTION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Load EfficientNet-B0
# ------------------------------------------------------------
feature_extractor = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

feature_extractor.trainable = False

print("EfficientNet-B0 loaded successfully")
print("Feature dimension:", feature_extractor.output_shape[-1])

# ------------------------------------------------------------
# 2. Data generators
# IMPORTANT: EfficientNet-B0 does NOT need rescale=1/255
# ------------------------------------------------------------
feature_datagen = ImageDataGenerator()

train_feature_generator = feature_datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

val_feature_generator = feature_datagen.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

test_feature_generator = feature_datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="class",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    classes=SELECTED_CLASSES,
    shuffle=False
)

# ------------------------------------------------------------
# 3. Extract features function
# ------------------------------------------------------------
def extract_features(generator):
    generator.reset()

    features = feature_extractor.predict(
        generator,
        verbose=1
    )

    labels = generator.classes.copy()

    return np.asarray(features), np.asarray(labels)


# ------------------------------------------------------------
# 4. Training features
# ------------------------------------------------------------
print("\n[1/3] Extracting training features...")

train_features, train_feature_labels = extract_features(
    train_feature_generator
)

print("Training features:", train_features.shape)
print("Training labels:", train_feature_labels.shape)


# ------------------------------------------------------------
# 5. Validation features
# ------------------------------------------------------------
print("\n[2/3] Extracting validation features...")

val_features, val_feature_labels = extract_features(
    val_feature_generator
)

print("Validation features:", val_features.shape)
print("Validation labels:", val_feature_labels.shape)


# ------------------------------------------------------------
# 6. Test features
# ------------------------------------------------------------
print("\n[3/3] Extracting test features...")

test_features, test_feature_labels = extract_features(
    test_feature_generator
)

print("Test features:", test_features.shape)
print("Test labels:", test_feature_labels.shape)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("FEATURE EXTRACTION COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Training   :", train_features.shape, train_feature_labels.shape)
print("Validation :", val_features.shape, val_feature_labels.shape)
print("Testing    :", test_features.shape, test_feature_labels.shape)

print("\nExpected:")
print("Training   : (1451, 1280) (1451,)")
print("Validation : (363, 1280)  (363,)")
print("Testing    : (80, 1280)   (80,)")

EFFICIENTNET-B0 DEEP FEATURE EXTRACTION
EfficientNet-B0 loaded successfully
Feature dimension: 1280
Found 1451 validated image filenames belonging to 5 classes.
Found 363 validated image filenames belonging to 5 classes.
Found 80 validated image filenames belonging to 5 classes.

[1/3] Extracting training features...
46/46 ━━━━━━━━━━━━━━━━━━━━ 32s 506ms/step
Training features: (1451, 1280)
Training labels: (1451,)

[2/3] Extracting validation features...
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 542ms/step
Validation features: (363, 1280)
Validation labels: (363,)

[3/3] Extracting test features...
3/3 ━━━━━━━━━━━━━━━━━━━━ 15s 7s/step
Test features: (80, 1280)
Test labels: (80,)

FEATURE EXTRACTION COMPLETED SUCCESSFULLY
Training   : (1451, 1280) (1451,)
Validation : (363, 1280) (363,)
Testing    : (80, 1280) (80,)

Expected:
Training   : (1451, 1280) (1451,)
Validation : (363, 1280)  (363,)
Testing    : (80, 1280)   (80,)


In [10]:
# ============================================================
# TABLE 2 - LOGISTIC REGRESSION
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("TABLE 2 - LOGISTIC REGRESSION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Scale deep features
# ------------------------------------------------------------
scaler = StandardScaler()

X_train_lr = scaler.fit_transform(train_features)
X_test_lr = scaler.transform(test_features)

y_train_lr = train_feature_labels
y_test_lr = test_feature_labels

print("Training data:", X_train_lr.shape)
print("Test data:", X_test_lr.shape)

# ------------------------------------------------------------
# 2. Train Logistic Regression
# ------------------------------------------------------------
print("\nTraining Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=2000,
    random_state=42,
    class_weight="balanced"
)

lr_model.fit(X_train_lr, y_train_lr)

print("Logistic Regression trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_lr = lr_model.predict(X_test_lr)
y_prob_lr = lr_model.predict_proba(X_test_lr)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
lr_accuracy = accuracy_score(y_test_lr, y_pred_lr)

lr_precision = precision_score(
    y_test_lr,
    y_pred_lr,
    average="weighted",
    zero_division=0
)

lr_recall = recall_score(
    y_test_lr,
    y_pred_lr,
    average="weighted",
    zero_division=0
)

lr_f1 = f1_score(
    y_test_lr,
    y_pred_lr,
    average="weighted",
    zero_division=0
)

y_test_onehot_lr = tf.keras.utils.to_categorical(
    y_test_lr,
    num_classes=NUM_CLASSES
)

lr_auc = roc_auc_score(
    y_test_onehot_lr,
    y_prob_lr,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 60)

print(f"Accuracy  : {lr_accuracy * 100:.2f}%")
print(f"Precision : {lr_precision * 100:.2f}%")
print(f"Recall    : {lr_recall * 100:.2f}%")
print(f"F1-Score  : {lr_f1 * 100:.2f}%")
print(f"AUC       : {lr_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Save result for Table 2
# ------------------------------------------------------------
table2_results = []

table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "Logistic Regression",
    "Accuracy (%)": lr_accuracy * 100,
    "Precision (%)": lr_precision * 100,
    "Recall (%)": lr_recall * 100,
    "F1-Score (%)": lr_f1 * 100,
    "AUC (%)": lr_auc * 100
})

print("\nTable 2 result saved successfully.")

TABLE 2 - LOGISTIC REGRESSION
Training data: (1451, 1280)
Test data: (80, 1280)

Training Logistic Regression...
Logistic Regression trained successfully!

LOGISTIC REGRESSION RESULTS
Accuracy  : 57.50%
Precision : 60.54%
Recall    : 57.50%
F1-Score  : 56.39%
AUC       : 83.71%

Table 2 result saved successfully.


In [11]:
# ============================================================
# TABLE 2 - DECISION TREE
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 60)
print("TABLE 2 - DECISION TREE")
print("=" * 60)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------
X_train_dt = train_features
X_test_dt = test_features

y_train_dt = train_feature_labels
y_test_dt = test_feature_labels

print("Training data:", X_train_dt.shape)
print("Test data:", X_test_dt.shape)

# ------------------------------------------------------------
# 2. Train Decision Tree
# ------------------------------------------------------------
print("\nTraining Decision Tree...")

dt_model = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

dt_model.fit(X_train_dt, y_train_dt)

print("Decision Tree trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_dt = dt_model.predict(X_test_dt)
y_prob_dt = dt_model.predict_proba(X_test_dt)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
dt_accuracy = accuracy_score(y_test_dt, y_pred_dt)

dt_precision = precision_score(
    y_test_dt,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

dt_recall = recall_score(
    y_test_dt,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

dt_f1 = f1_score(
    y_test_dt,
    y_pred_dt,
    average="weighted",
    zero_division=0
)

y_test_onehot_dt = tf.keras.utils.to_categorical(
    y_test_dt,
    num_classes=NUM_CLASSES
)

dt_auc = roc_auc_score(
    y_test_onehot_dt,
    y_prob_dt,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("DECISION TREE RESULTS")
print("=" * 60)

print(f"Accuracy  : {dt_accuracy * 100:.2f}%")
print(f"Precision : {dt_precision * 100:.2f}%")
print(f"Recall    : {dt_recall * 100:.2f}%")
print(f"F1-Score  : {dt_f1 * 100:.2f}%")
print(f"AUC       : {dt_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "Decision Tree",
    "Accuracy (%)": dt_accuracy * 100,
    "Precision (%)": dt_precision * 100,
    "Recall (%)": dt_recall * 100,
    "F1-Score (%)": dt_f1 * 100,
    "AUC (%)": dt_auc * 100
})

print("\nDecision Tree result added to Table 2.")

TABLE 2 - DECISION TREE
Training data: (1451, 1280)
Test data: (80, 1280)

Training Decision Tree...
Decision Tree trained successfully!

DECISION TREE RESULTS
Accuracy  : 38.75%
Precision : 38.20%
Recall    : 38.75%
F1-Score  : 38.11%
AUC       : 61.72%

Decision Tree result added to Table 2.


In [12]:
# ============================================================
# TABLE 2 - RANDOM FOREST
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 60)
print("TABLE 2 - RANDOM FOREST")
print("=" * 60)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------
X_train_rf = train_features
X_test_rf = test_features

y_train_rf = train_feature_labels
y_test_rf = test_feature_labels

print("Training data:", X_train_rf.shape)
print("Test data:", X_test_rf.shape)

# ------------------------------------------------------------
# 2. Train Random Forest
# ------------------------------------------------------------
print("\nTraining Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train_rf)

print("Random Forest trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_rf = rf_model.predict(X_test_rf)
y_prob_rf = rf_model.predict_proba(X_test_rf)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
rf_accuracy = accuracy_score(y_test_rf, y_pred_rf)

rf_precision = precision_score(
    y_test_rf,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

rf_recall = recall_score(
    y_test_rf,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

rf_f1 = f1_score(
    y_test_rf,
    y_pred_rf,
    average="weighted",
    zero_division=0
)

y_test_onehot_rf = tf.keras.utils.to_categorical(
    y_test_rf,
    num_classes=NUM_CLASSES
)

rf_auc = roc_auc_score(
    y_test_onehot_rf,
    y_prob_rf,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("RANDOM FOREST RESULTS")
print("=" * 60)

print(f"Accuracy  : {rf_accuracy * 100:.2f}%")
print(f"Precision : {rf_precision * 100:.2f}%")
print(f"Recall    : {rf_recall * 100:.2f}%")
print(f"F1-Score  : {rf_f1 * 100:.2f}%")
print(f"AUC       : {rf_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "Random Forest",
    "Accuracy (%)": rf_accuracy * 100,
    "Precision (%)": rf_precision * 100,
    "Recall (%)": rf_recall * 100,
    "F1-Score (%)": rf_f1 * 100,
    "AUC (%)": rf_auc * 100
})

print("\nRandom Forest result added to Table 2.")

TABLE 2 - RANDOM FOREST
Training data: (1451, 1280)
Test data: (80, 1280)

Training Random Forest...
Random Forest trained successfully!

RANDOM FOREST RESULTS
Accuracy  : 47.50%
Precision : 48.59%
Recall    : 47.50%
F1-Score  : 44.83%
AUC       : 85.97%

Random Forest result added to Table 2.


In [13]:
# ============================================================
# TABLE 2 - K-NEAREST NEIGHBORS (KNN)
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("TABLE 2 - K-NEAREST NEIGHBORS (KNN)")
print("=" * 60)

# ------------------------------------------------------------
# 1. Scale deep features
# ------------------------------------------------------------
scaler_knn = StandardScaler()

X_train_knn = scaler_knn.fit_transform(train_features)
X_test_knn = scaler_knn.transform(test_features)

y_train_knn = train_feature_labels
y_test_knn = test_feature_labels

print("Training data:", X_train_knn.shape)
print("Test data:", X_test_knn.shape)

# ------------------------------------------------------------
# 2. Train KNN
# ------------------------------------------------------------
print("\nTraining KNN...")

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    n_jobs=-1
)

knn_model.fit(X_train_knn, y_train_knn)

print("KNN trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_knn = knn_model.predict(X_test_knn)
y_prob_knn = knn_model.predict_proba(X_test_knn)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
knn_accuracy = accuracy_score(y_test_knn, y_pred_knn)

knn_precision = precision_score(
    y_test_knn,
    y_pred_knn,
    average="weighted",
    zero_division=0
)

knn_recall = recall_score(
    y_test_knn,
    y_pred_knn,
    average="weighted",
    zero_division=0
)

knn_f1 = f1_score(
    y_test_knn,
    y_pred_knn,
    average="weighted",
    zero_division=0
)

y_test_onehot_knn = tf.keras.utils.to_categorical(
    y_test_knn,
    num_classes=NUM_CLASSES
)

knn_auc = roc_auc_score(
    y_test_onehot_knn,
    y_prob_knn,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("KNN RESULTS")
print("=" * 60)

print(f"Accuracy  : {knn_accuracy * 100:.2f}%")
print(f"Precision : {knn_precision * 100:.2f}%")
print(f"Recall    : {knn_recall * 100:.2f}%")
print(f"F1-Score  : {knn_f1 * 100:.2f}%")
print(f"AUC       : {knn_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "K-Nearest Neighbors (KNN)",
    "Accuracy (%)": knn_accuracy * 100,
    "Precision (%)": knn_precision * 100,
    "Recall (%)": knn_recall * 100,
    "F1-Score (%)": knn_f1 * 100,
    "AUC (%)": knn_auc * 100
})

print("\nKNN result added to Table 2.")

TABLE 2 - K-NEAREST NEIGHBORS (KNN)
Training data: (1451, 1280)
Test data: (80, 1280)

Training KNN...
KNN trained successfully!

KNN RESULTS
Accuracy  : 41.25%
Precision : 42.87%
Recall    : 41.25%
F1-Score  : 39.83%
AUC       : 73.64%

KNN result added to Table 2.


In [14]:
# ============================================================
# TABLE 2 - LINEAR SVM
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("TABLE 2 - LINEAR SVM")
print("=" * 60)

# ------------------------------------------------------------
# 1. Scale deep features
# ------------------------------------------------------------
scaler_linear_svm = StandardScaler()

X_train_svm = scaler_linear_svm.fit_transform(train_features)
X_test_svm = scaler_linear_svm.transform(test_features)

y_train_svm = train_feature_labels
y_test_svm = test_feature_labels

print("Training data:", X_train_svm.shape)
print("Test data:", X_test_svm.shape)

# ------------------------------------------------------------
# 2. Train Linear SVM
# ------------------------------------------------------------
print("\nTraining Linear SVM...")

linear_svm_model = SVC(
    kernel="linear",
    probability=True,
    class_weight="balanced",
    random_state=42
)

linear_svm_model.fit(X_train_svm, y_train_svm)

print("Linear SVM trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_linear_svm = linear_svm_model.predict(X_test_svm)
y_prob_linear_svm = linear_svm_model.predict_proba(X_test_svm)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
linear_svm_accuracy = accuracy_score(
    y_test_svm, y_pred_linear_svm
)

linear_svm_precision = precision_score(
    y_test_svm,
    y_pred_linear_svm,
    average="weighted",
    zero_division=0
)

linear_svm_recall = recall_score(
    y_test_svm,
    y_pred_linear_svm,
    average="weighted",
    zero_division=0
)

linear_svm_f1 = f1_score(
    y_test_svm,
    y_pred_linear_svm,
    average="weighted",
    zero_division=0
)

y_test_onehot_linear_svm = tf.keras.utils.to_categorical(
    y_test_svm,
    num_classes=NUM_CLASSES
)

linear_svm_auc = roc_auc_score(
    y_test_onehot_linear_svm,
    y_prob_linear_svm,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("LINEAR SVM RESULTS")
print("=" * 60)

print(f"Accuracy  : {linear_svm_accuracy * 100:.2f}%")
print(f"Precision : {linear_svm_precision * 100:.2f}%")
print(f"Recall    : {linear_svm_recall * 100:.2f}%")
print(f"F1-Score  : {linear_svm_f1 * 100:.2f}%")
print(f"AUC       : {linear_svm_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "Linear SVM",
    "Accuracy (%)": linear_svm_accuracy * 100,
    "Precision (%)": linear_svm_precision * 100,
    "Recall (%)": linear_svm_recall * 100,
    "F1-Score (%)": linear_svm_f1 * 100,
    "AUC (%)": linear_svm_auc * 100
})

print("\nLinear SVM result added to Table 2.")

TABLE 2 - LINEAR SVM
Training data: (1451, 1280)
Test data: (80, 1280)

Training Linear SVM...
Linear SVM trained successfully!

LINEAR SVM RESULTS
Accuracy  : 60.00%
Precision : 62.27%
Recall    : 60.00%
F1-Score  : 59.07%
AUC       : 86.37%

Linear SVM result added to Table 2.


In [15]:
# ============================================================
# TABLE 2 - RBF-SVM
# EfficientNet-B0 Deep Features
# ============================================================

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("TABLE 2 - RBF-SVM")
print("=" * 60)

# ------------------------------------------------------------
# 1. Scale deep features
# ------------------------------------------------------------
scaler_rbf_svm = StandardScaler()

X_train_rbf = scaler_rbf_svm.fit_transform(train_features)
X_test_rbf = scaler_rbf_svm.transform(test_features)

y_train_rbf = train_feature_labels
y_test_rbf = test_feature_labels

print("Training data:", X_train_rbf.shape)
print("Test data:", X_test_rbf.shape)

# ------------------------------------------------------------
# 2. Train RBF-SVM
# ------------------------------------------------------------
print("\nTraining RBF-SVM...")

rbf_svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    probability=True,
    class_weight="balanced",
    random_state=42
)

rbf_svm_model.fit(X_train_rbf, y_train_rbf)

print("RBF-SVM trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_rbf = rbf_svm_model.predict(X_test_rbf)
y_prob_rbf = rbf_svm_model.predict_proba(X_test_rbf)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
rbf_accuracy = accuracy_score(
    y_test_rbf, y_pred_rbf
)

rbf_precision = precision_score(
    y_test_rbf,
    y_pred_rbf,
    average="weighted",
    zero_division=0
)

rbf_recall = recall_score(
    y_test_rbf,
    y_pred_rbf,
    average="weighted",
    zero_division=0
)

rbf_f1 = f1_score(
    y_test_rbf,
    y_pred_rbf,
    average="weighted",
    zero_division=0
)

y_test_onehot_rbf = tf.keras.utils.to_categorical(
    y_test_rbf,
    num_classes=NUM_CLASSES
)

rbf_auc = roc_auc_score(
    y_test_onehot_rbf,
    y_prob_rbf,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("RBF-SVM RESULTS")
print("=" * 60)

print(f"Accuracy  : {rbf_accuracy * 100:.2f}%")
print(f"Precision : {rbf_precision * 100:.2f}%")
print(f"Recall    : {rbf_recall * 100:.2f}%")
print(f"F1-Score  : {rbf_f1 * 100:.2f}%")
print(f"AUC       : {rbf_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "RBF-SVM",
    "Accuracy (%)": rbf_accuracy * 100,
    "Precision (%)": rbf_precision * 100,
    "Recall (%)": rbf_recall * 100,
    "F1-Score (%)": rbf_f1 * 100,
    "AUC (%)": rbf_auc * 100
})

print("\nRBF-SVM result added to Table 2.")

TABLE 2 - RBF-SVM
Training data: (1451, 1280)
Test data: (80, 1280)

Training RBF-SVM...
RBF-SVM trained successfully!

RBF-SVM RESULTS
Accuracy  : 56.25%
Precision : 58.36%
Recall    : 56.25%
F1-Score  : 54.90%
AUC       : 87.60%

RBF-SVM result added to Table 2.


In [16]:
# ============================================================
# TABLE 2 - XGBOOST
# EfficientNet-B0 Deep Features
# ============================================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 60)
print("TABLE 2 - XGBOOST")
print("=" * 60)

# ------------------------------------------------------------
# 1. Prepare data
# ------------------------------------------------------------
X_train_xgb = train_features
X_test_xgb = test_features

y_train_xgb = train_feature_labels
y_test_xgb = test_feature_labels

print("Training data:", X_train_xgb.shape)
print("Test data:", X_test_xgb.shape)

# ------------------------------------------------------------
# 2. Train XGBoost
# ------------------------------------------------------------
print("\nTraining XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb
)

print("XGBoost trained successfully!")

# ------------------------------------------------------------
# 3. Predictions
# ------------------------------------------------------------
y_pred_xgb = xgb_model.predict(X_test_xgb)
y_prob_xgb = xgb_model.predict_proba(X_test_xgb)

# Make sure predictions are integer labels
y_pred_xgb = np.asarray(y_pred_xgb).astype(int)

# ------------------------------------------------------------
# 4. Calculate metrics
# ------------------------------------------------------------
xgb_accuracy = accuracy_score(
    y_test_xgb,
    y_pred_xgb
)

xgb_precision = precision_score(
    y_test_xgb,
    y_pred_xgb,
    average="weighted",
    zero_division=0
)

xgb_recall = recall_score(
    y_test_xgb,
    y_pred_xgb,
    average="weighted",
    zero_division=0
)

xgb_f1 = f1_score(
    y_test_xgb,
    y_pred_xgb,
    average="weighted",
    zero_division=0
)

y_test_onehot_xgb = tf.keras.utils.to_categorical(
    y_test_xgb,
    num_classes=NUM_CLASSES
)

xgb_auc = roc_auc_score(
    y_test_onehot_xgb,
    y_prob_xgb,
    multi_class="ovr",
    average="weighted"
)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("XGBOOST RESULTS")
print("=" * 60)

print(f"Accuracy  : {xgb_accuracy * 100:.2f}%")
print(f"Precision : {xgb_precision * 100:.2f}%")
print(f"Recall    : {xgb_recall * 100:.2f}%")
print(f"F1-Score  : {xgb_f1 * 100:.2f}%")
print(f"AUC       : {xgb_auc * 100:.2f}%")

# ------------------------------------------------------------
# 6. Add result to Table 2
# ------------------------------------------------------------
table2_results.append({
    "Feature Extractor": "EfficientNet-B0 Deep Features",
    "Classifier": "XGBoost",
    "Accuracy (%)": xgb_accuracy * 100,
    "Precision (%)": xgb_precision * 100,
    "Recall (%)": xgb_recall * 100,
    "F1-Score (%)": xgb_f1 * 100,
    "AUC (%)": xgb_auc * 100
})

print("\nXGBoost result added to Table 2.")

TABLE 2 - XGBOOST
Training data: (1451, 1280)
Test data: (80, 1280)

Training XGBoost...
XGBoost trained successfully!

XGBOOST RESULTS
Accuracy  : 56.25%
Precision : 58.41%
Recall    : 56.25%
F1-Score  : 53.91%
AUC       : 83.40%

XGBoost result added to Table 2.


In [17]:
# ============================================================
# TABLE 2 - COMPLETE RESULTS
# ============================================================

print("=" * 90)
print("TABLE 2 - COMPARISON OF DIFFERENT CLASSIFIERS")
print("Feature Extractor: EfficientNet-B0 Deep Features")
print("=" * 90)

# Convert results to DataFrame
table2_df = pd.DataFrame(table2_results)

# Keep correct order
classifier_order = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "K-Nearest Neighbors (KNN)",
    "Linear SVM",
    "RBF-SVM",
    "XGBoost"
]

table2_df["Classifier"] = pd.Categorical(
    table2_df["Classifier"],
    categories=classifier_order,
    ordered=True
)

table2_df = table2_df.sort_values("Classifier").reset_index(drop=True)

# Round values
metric_columns = [
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-Score (%)",
    "AUC (%)"
]

table2_df[metric_columns] = table2_df[metric_columns].round(2)

# Display final table
display(table2_df)

# ------------------------------------------------------------
# Compact table for assignment
# ------------------------------------------------------------
table2_assignment = table2_df[
    [
        "Feature Extractor",
        "Classifier",
        "Accuracy (%)",
        "Precision (%)",
        "Recall (%)",
        "F1-Score (%)",
        "AUC (%)"
    ]
].copy()

print("\n" + "=" * 90)
print("TABLE 2 - ASSIGNMENT FORMAT")
print("=" * 90)

display(table2_assignment)

TABLE 2 - COMPARISON OF DIFFERENT CLASSIFIERS
Feature Extractor: EfficientNet-B0 Deep Features


,Feature Extractor,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,EfficientNet-B0 Deep Features,Logistic Regression,57.50,60.54,57.50,56.39,83.71
1,EfficientNet-B0 Deep Features,Decision Tree,38.75,38.20,38.75,38.11,61.72
2,EfficientNet-B0 Deep Features,Random Forest,47.50,48.59,47.50,44.83,85.97
3,EfficientNet-B0 Deep Features,K-Nearest Neighbors (KNN),41.25,42.87,41.25,39.83,73.64
4,EfficientNet-B0 Deep Features,Linear SVM,60.00,62.27,60.00,59.07,86.37
5,EfficientNet-B0 Deep Features,RBF-SVM,56.25,58.36,56.25,54.90,87.60
6,EfficientNet-B0 Deep Features,XGBoost,56.25,58.41,56.25,53.91,83.40



TABLE 2 - ASSIGNMENT FORMAT


,Feature Extractor,Classifier,Accuracy (%),Precision (%),Recall (%),F1-Score (%),AUC (%)
0,EfficientNet-B0 Deep Features,Logistic Regression,57.50,60.54,57.50,56.39,83.71
1,EfficientNet-B0 Deep Features,Decision Tree,38.75,38.20,38.75,38.11,61.72
2,EfficientNet-B0 Deep Features,Random Forest,47.50,48.59,47.50,44.83,85.97
3,EfficientNet-B0 Deep Features,K-Nearest Neighbors (KNN),41.25,42.87,41.25,39.83,73.64
4,EfficientNet-B0 Deep Features,Linear SVM,60.00,62.27,60.00,59.07,86.37
5,EfficientNet-B0 Deep Features,RBF-SVM,56.25,58.36,56.25,54.90,87.60
6,EfficientNet-B0 Deep Features,XGBoost,56.25,58.41,56.25,53.91,83.40


Architecture metrics

In [18]:
# ============================================================
# TABLE 3 - COMPUTATIONAL EFFICIENCY
# STEP 1: MODEL ARCHITECTURE METRICS
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd
import os

from tensorflow.keras import layers, models
from tensorflow.keras.applications import (
    VGG16,
    VGG19,
    ResNet50,
    DenseNet121,
    EfficientNetB0
)

NUM_CLASSES = 5
INPUT_SHAPE = (224, 224, 3)


# ------------------------------------------------------------
# 1. AlexNet
# ------------------------------------------------------------
def build_alexnet():
    model = models.Sequential([
        layers.Input(shape=INPUT_SHAPE),

        layers.Conv2D(96, (11, 11), strides=4, activation="relu"),
        layers.MaxPooling2D((3, 3), strides=2),

        layers.Conv2D(256, (5, 5), padding="same", activation="relu"),
        layers.MaxPooling2D((3, 3), strides=2),

        layers.Conv2D(384, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(384, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((3, 3), strides=2),

        layers.Flatten(),
        layers.Dense(4096, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(4096, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ------------------------------------------------------------
# 2. VGG16
# ------------------------------------------------------------
def build_vgg16():
    base = VGG16(
        weights=None,
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ------------------------------------------------------------
# 3. VGG19
# ------------------------------------------------------------
def build_vgg19():
    base = VGG19(
        weights=None,
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ------------------------------------------------------------
# 4. ResNet18
# ------------------------------------------------------------
def residual_block(x, filters, stride=1):

    shortcut = x

    x = layers.Conv2D(
        filters,
        3,
        strides=stride,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        use_bias=False
    )(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters or stride != 1:
        shortcut = layers.Conv2D(
            filters,
            1,
            strides=stride,
            padding="same",
            use_bias=False
        )(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


def build_resnet18():

    inputs = layers.Input(shape=INPUT_SHAPE)

    x = layers.Conv2D(
        64,
        7,
        strides=2,
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    # 2 blocks
    x = residual_block(x, 64)
    x = residual_block(x, 64)

    # 2 blocks
    x = residual_block(x, 128, stride=2)
    x = residual_block(x, 128)

    # 2 blocks
    x = residual_block(x, 256, stride=2)
    x = residual_block(x, 256)

    # 2 blocks
    x = residual_block(x, 512, stride=2)
    x = residual_block(x, 512)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    return models.Model(inputs, outputs)


# ------------------------------------------------------------
# 5. ResNet50
# ------------------------------------------------------------
def build_resnet50():
    base = ResNet50(
        weights=None,
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ------------------------------------------------------------
# 6. DenseNet121
# ------------------------------------------------------------
def build_densenet121():
    base = DenseNet121(
        weights=None,
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ------------------------------------------------------------
# 7. EfficientNet-B0
# ------------------------------------------------------------
def build_efficientnet():
    base = EfficientNetB0(
        weights=None,
        include_top=False,
        input_shape=INPUT_SHAPE
    )

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    return model


# ============================================================
# CREATE MODELS
# ============================================================

print("Building models...")

models_dict = {
    "AlexNet": build_alexnet(),
    "VGG16": build_vgg16(),
    "VGG19": build_vgg19(),
    "ResNet18": build_resnet18(),
    "ResNet50": build_resnet50(),
    "DenseNet121": build_densenet121(),
    "EfficientNet-B0": build_efficientnet()
}

print("\nAll models built successfully!\n")


# ============================================================
# PARAMETER + MODEL SIZE
# ============================================================

architecture_results = []

for model_name, model in models_dict.items():

    params = model.count_params()

    # 4 bytes per parameter for float32
    model_size_mb = (params * 4) / (1024 ** 2)

    architecture_results.append({
        "Model": model_name,
        "Parameters (M)": params / 1e6,
        "Model Size (MB)": model_size_mb
    })


architecture_df = pd.DataFrame(architecture_results)

architecture_df["Parameters (M)"] = architecture_df[
    "Parameters (M)"
].round(2)

architecture_df["Model Size (MB)"] = architecture_df[
    "Model Size (MB)"
].round(2)


print("=" * 70)
print("PARAMETERS AND MODEL SIZE")
print("=" * 70)

display(architecture_df)

Building models...

All models built successfully!

PARAMETERS AND MODEL SIZE


,Model,Parameters (M),Model Size (MB)
0,AlexNet,46.77,178.40
1,VGG16,14.72,56.14
2,VGG19,20.03,76.40
3,ResNet18,11.19,42.68
4,ResNet50,23.60,90.02
5,DenseNet121,7.04,26.87
6,EfficientNet-B0,4.06,15.47


In [19]:
# ============================================================
# TABLE 3 - STEP 2: FLOPs CALCULATION
# ============================================================

!pip -q install thop

from thop import profile
import torch
import torch.nn as nn

print("=" * 70)
print("CALCULATING FLOPs")
print("=" * 70)


# ------------------------------------------------------------
# Convert Keras model to approximate FLOPs using TensorFlow
# profiler
# ------------------------------------------------------------

def calculate_flops_tf(model):

    @tf.function
    def forward(x):
        return model(x, training=False)

    concrete_func = forward.get_concrete_function(
        tf.TensorSpec(
            [1, 224, 224, 3],
            tf.float32
        )
    )

    frozen_func = convert_variables_to_constants_v2(
        concrete_func
    )

    graph_def = frozen_func.graph.as_graph_def()

    with tf.Graph().as_default() as graph:

        tf.graph_util.import_graph_def(
            graph_def,
            name=""
        )

        run_meta = tf.compat.v1.RunMetadata()

        opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

        flops = tf.compat.v1.profiler.profile(
            graph=graph,
            run_meta=run_meta,
            cmd="op",
            options=opts
        )

    return flops.total_float_ops


# ------------------------------------------------------------
# Import TensorFlow profiler utility
# ------------------------------------------------------------

from tensorflow.python.framework.convert_to_constants import (
    convert_variables_to_constants_v2
)


# ------------------------------------------------------------
# Calculate FLOPs for all models
# ------------------------------------------------------------

flops_results = []

for model_name, model in models_dict.items():

    print(f"\nCalculating FLOPs for {model_name}...")

    try:
        flops = calculate_flops_tf(model)

        flops_g = flops / 1e9

        print(f"{model_name}: {flops_g:.3f} G FLOPs")

        flops_results.append({
            "Model": model_name,
            "FLOPs (G)": flops_g
        })

    except Exception as e:

        print(f"Error for {model_name}: {e}")

        flops_results.append({
            "Model": model_name,
            "FLOPs (G)": np.nan
        })


# ------------------------------------------------------------
# Create FLOPs DataFrame
# ------------------------------------------------------------

flops_df = pd.DataFrame(flops_results)

flops_df["FLOPs (G)"] = flops_df[
    "FLOPs (G)"
].round(3)


print("\n" + "=" * 70)
print("FLOPs RESULTS")
print("=" * 70)

display(flops_df)

CALCULATING FLOPs

Calculating FLOPs for AlexNet...


Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


AlexNet: 2.013 G FLOPs

Calculating FLOPs for VGG16...
VGG16: 30.713 G FLOPs

Calculating FLOPs for VGG19...
VGG19: 39.038 G FLOPs

Calculating FLOPs for ResNet18...
ResNet18: 3.635 G FLOPs

Calculating FLOPs for ResNet50...
ResNet50: 7.751 G FLOPs

Calculating FLOPs for DenseNet121...
DenseNet121: 5.700 G FLOPs

Calculating FLOPs for EfficientNet-B0...
EfficientNet-B0: 0.800 G FLOPs

FLOPs RESULTS


,Model,FLOPs (G)
0,AlexNet,2.013
1,VGG16,30.713
2,VGG19,39.038
3,ResNet18,3.635
4,ResNet50,7.751
5,DenseNet121,5.700
6,EfficientNet-B0,0.800


In [20]:
# ============================================================
# TABLE 3 - COMPLETE COMPUTATIONAL EFFICIENCY COMPARISON
# ============================================================

import time
import numpy as np
import pandas as pd
import tensorflow as tf

print("=" * 100)
print("TABLE 3 - COMPUTATIONAL EFFICIENCY COMPARISON")
print("=" * 100)


# ------------------------------------------------------------
# 1. Accuracy values from Table 1
# ------------------------------------------------------------

accuracy_table1 = {
    "AlexNet": 56.25,
    "VGG16": 51.25,
    "VGG19": 48.75,
    "ResNet18": 60.00,
    "ResNet50": 57.50,
    "DenseNet121": 55.00,
    "EfficientNet-B0": 62.50
}


# ------------------------------------------------------------
# 2. Model order required by assignment
# ------------------------------------------------------------

model_order = [
    "AlexNet",
    "VGG16",
    "VGG19",
    "ResNet18",
    "ResNet50",
    "DenseNet121",
    "EfficientNet-B0"
]


# ------------------------------------------------------------
# 3. Calculate inference time
#    Batch size = 1
# ------------------------------------------------------------

print("\nCalculating inference time...\n")

inference_results = []

for model_name in model_order:

    model = models_dict[model_name]

    # Dummy input
    dummy_input = tf.random.uniform(
        shape=(1, 224, 224, 3),
        dtype=tf.float32
    )

    # Warm-up
    for _ in range(3):
        _ = model(dummy_input, training=False)

    # Measure inference
    times = []

    for _ in range(10):

        start = time.perf_counter()

        _ = model(dummy_input, training=False)

        # Ensure GPU operation is completed
        if tf.config.list_physical_devices("GPU"):
            _ = tf.reduce_sum(_).numpy()

        end = time.perf_counter()

        times.append((end - start) * 1000)

    # Median inference time
    inference_time_ms = np.median(times)

    inference_results.append({
        "Model": model_name,
        "Inference Time (ms)": inference_time_ms
    })

    print(
        f"{model_name:20s} : "
        f"{inference_time_ms:.2f} ms"
    )


inference_df = pd.DataFrame(inference_results)


# ------------------------------------------------------------
# 4. Check architecture results
# ------------------------------------------------------------

architecture_final = architecture_df[
    [
        "Model",
        "Parameters (M)",
        "Model Size (MB)"
    ]
].copy()


# ------------------------------------------------------------
# 5. Check FLOPs results
# ------------------------------------------------------------

flops_final = flops_df[
    [
        "Model",
        "FLOPs (G)"
    ]
].copy()


# ------------------------------------------------------------
# 6. Combine everything
# ------------------------------------------------------------

table3_df = architecture_final.merge(
    flops_final,
    on="Model",
    how="left"
)

table3_df = table3_df.merge(
    inference_df,
    on="Model",
    how="left"
)


# ------------------------------------------------------------
# 7. Add accuracy
# ------------------------------------------------------------

table3_df["Accuracy (%)"] = table3_df[
    "Model"
].map(accuracy_table1)


# ------------------------------------------------------------
# 8. Correct model order
# ------------------------------------------------------------

table3_df["Model"] = pd.Categorical(
    table3_df["Model"],
    categories=model_order,
    ordered=True
)

table3_df = table3_df.sort_values(
    "Model"
).reset_index(drop=True)


# ------------------------------------------------------------
# 9. Round values
# ------------------------------------------------------------

table3_df["Parameters (M)"] = table3_df[
    "Parameters (M)"
].round(2)

table3_df["Model Size (MB)"] = table3_df[
    "Model Size (MB)"
].round(2)

table3_df["FLOPs (G)"] = table3_df[
    "FLOPs (G)"
].round(3)

table3_df["Inference Time (ms)"] = table3_df[
    "Inference Time (ms)"
].round(2)

table3_df["Accuracy (%)"] = table3_df[
    "Accuracy (%)"
].round(2)


# ------------------------------------------------------------
# 10. Final assignment table
# ------------------------------------------------------------

table3_df = table3_df[
    [
        "Model",
        "Parameters (M)",
        "Model Size (MB)",
        "FLOPs (G)",
        "Inference Time (ms)",
        "Accuracy (%)"
    ]
]


print("\n")
print("=" * 100)
print("FINAL TABLE 3")
print("=" * 100)

display(table3_df)


# ------------------------------------------------------------
# 11. Save CSV
# ------------------------------------------------------------

table3_df.to_csv(
    "Table_3_Computational_Efficiency.csv",
    index=False
)

print("\nTable 3 saved as: Table_3_Computational_Efficiency.csv")

TABLE 3 - COMPUTATIONAL EFFICIENCY COMPARISON

Calculating inference time...

AlexNet              : 24.30 ms
VGG16                : 45.61 ms
VGG19                : 59.44 ms
ResNet18             : 107.82 ms
ResNet50             : 558.92 ms
DenseNet121          : 518.57 ms
EfficientNet-B0      : 330.92 ms


FINAL TABLE 3


,Model,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
0,AlexNet,46.77,178.40,2.013,24.30,56.25
1,VGG16,14.72,56.14,30.713,45.61,51.25
2,VGG19,20.03,76.40,39.038,59.44,48.75
3,ResNet18,11.19,42.68,3.635,107.82,60.00
4,ResNet50,23.60,90.02,7.751,558.92,57.50
5,DenseNet121,7.04,26.87,5.700,518.57,55.00
6,EfficientNet-B0,4.06,15.47,0.800,330.92,62.50



Table 3 saved as: Table_3_Computational_Efficiency.csv
